# Рисунки для встречи со Сколтехом, 25 ноября 2026 г.

Выполните ячейки в проектной .venv с локальным каталогом runs. Затем из каталога встречи выполните `latexmk talk.tex`. Для сборки слайдов достаточно сохраненных рисунков.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from scipy.integrate import solve_ivp

from lle.analysis.criteria import is_homogeneous, is_stationary
from lle.analysis.fields import relative_l2_error
from lle.analysis.peaks import peak_indices
from lle.storage import load_evolution

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "experiments/01_initial-pulse-outcomes").is_dir())
figures = root / "publications/2026-11-25_skoltech/figures"
figures.mkdir(exist_ok=True)
runs = root / "runs"
teal, orange, gray = "#087b83", "#b85726", "#58677a"
plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})

def export(fig, name):
    fig.savefig(figures / f"{name}.pdf", dpi=220, bbox_inches="tight")
    plt.show()

## Амплитуда и ширина начального импульса

Эксперимент 01. Для классификации используем снимки на [80, 100] и критерии исходного блокнота. Для двух профилей выбираем одну ширину и разные амплитуды.

In [ ]:
def classify(tail):
    kind = "Прочее"
    if is_stationary(tail):
        if all(is_homogeneous(p) for p in tail.intensity):
            kind = "Однородное"
        elif all(not is_homogeneous(p) and len(peak_indices(p)) == 1 for p in tail.intensity):
            kind = "Один пик"
    return kind


fig, (ax, profile) = plt.subplots(1, 2, figsize=(10, 3.4), layout="constrained")
groups = {"Однородное": [], "Один пик": [], "Прочее": []}
manifest = root / "experiments/01_initial-pulse-outcomes/coarse_runs.csv"
with manifest.open() as stream:
    rows = list(csv.DictReader(stream))
for row in rows:
    field = load_evolution(runs / row["run_id"])
    tail = field[(field.times >= 80) & (field.times <= 100)]
    kind = classify(tail)
    width, amplitude = float(row["width"]), float(row["amplitude"])
    groups[kind].append((width, amplitude))
    if np.isclose(width, 0.3293312) and amplitude in (0.4, 1.6):
        color = gray if amplitude == 0.4 else teal
        ax.scatter(width, amplitude, s=180, facecolors="none", edgecolors=color)
        profile.plot(field.theta, tail[-1].intensity.values, color=color, label=f"A = {amplitude:g}")
for (label, points), color, marker in zip(groups.items(), (gray, teal, orange), ("o", "s", "^")):
    ax.scatter(*np.array(points).T, c=color, marker=marker, s=30, label=f"{label} ({len(points)})")
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel="Амплитуда A")
ax.set_xticks([0.05, 0.1, 0.2, 0.4, 0.7], labels=["0.05", "0.1", "0.2", "0.4", "0.7"])
ax.minorticks_off()
ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(0, 1.28), ncols=3, frameon=False)
profile.set(xlabel=r"Координата $\theta$", ylabel=r"Интенсивность $|\psi|^2$", title=r"Профили при $t=100$, $\sigma=0.329$")
profile.legend(frameon=False)
print({k: len(v) for k, v in groups.items()})
export(fig, "outcomes")

## Серии по накачке и отстройке

Эксперимент 03. Каждый узел соответствует расчету при тех же начальных амплитуде и ширине. Сравниваем поле на окне [80, 100].

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from lle.viz.plots import cell_edges

experiment = root / "experiments/03_equation-outcome-maps"
maps = {}
for filename in ("baseline_runs.csv", "map_runs.csv"):
    with (experiment / filename).open(encoding="utf8") as stream:
        for row in csv.DictReader(stream):
            key = float(row["F"]), float(row["delta"])
            node = round(float(row["width"]), 12), round(float(row["amplitude"]), 12)
            field = load_evolution(runs / row["run_id"])
            tail = field[(field.times >= 80) & (field.times <= 100)]
            np.testing.assert_allclose(tail.times[[0, -1]], [80, 100])
            maps.setdefault(key, {})[node] = classify(tail)
baseline_nodes = maps[2, 3].keys()
assert len(baseline_nodes) == 64 and all(m.keys() == baseline_nodes for m in maps.values())
colors = {"Однородное": "#c9cdd1", "Один пик": "#007f86", "Прочее": "#d66926"}
codes = {label: index for index, label in enumerate(colors)}
widths = sorted({width for width, amplitude in baseline_nodes})
amplitudes = sorted({amplitude for width, amplitude in baseline_nodes})
width_edges = cell_edges(widths, logarithmic=True)
amplitude_edges = cell_edges(amplitudes)

def plot_series(cases, titles, name):
    fig, axes = plt.subplots(1, 3, figsize=(10.6, 3.8), sharex=True, sharey=True, layout="constrained")
    for ax, key, title in zip(axes, cases, titles, strict=True):
        grid = [[codes[maps[key][width, amplitude]] for width in widths] for amplitude in amplitudes]
        ax.pcolormesh(width_edges, amplitude_edges, grid, cmap=ListedColormap(list(colors.values())),
                      vmin=-0.5, vmax=len(colors)-0.5, edgecolors="white", linewidth=1.6)
        ax.set(xscale="log", title=title, xlabel=r"Ширина $\sigma$")
        ax.set_xticks([0.05, 0.1, 0.2, 0.4, 0.7], labels=["0,05", "0,1", "0,2", "0,4", "0,7"])
        ax.minorticks_off()
        print(key, {label: list(maps[key].values()).count(label) for label in colors})
    axes[0].set_ylabel("Амплитуда A")
    handles = [Patch(facecolor=color, label=label) for label, color in colors.items()]
    fig.legend(handles=handles, loc="outside lower center", ncols=3, frameon=False)
    export(fig, name)

plot_series([(1.8, 3), (2, 3), (2.2, 3)], ["F = 1,8", "F = 2", "F = 2,2"], "pump-series")
plot_series([(2, 2.5), (2, 3), (2, 3.5)], [r"$\delta$ = 2,5", r"$\delta$ = 3", r"$\delta$ = 3,5"], "detuning-series")

## Динамика при A = 2,8

Два запуска эксперимента 02. Показываем центральную часть кольца на общей шкале интенсивности.

In [ ]:
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(9, 4.4), layout="constrained")
for ax, run_id, width in zip(
    axes,
    ("2026-11-02_09-28-14_701115", "2026-11-02_09-26-12_962949"),
    ("0,480", "0,700"),
):
    field = load_evolution(runs / run_id)
    mask = abs(field.theta) <= 1.45
    image = ax.pcolormesh(field.theta[mask], field.times, abs(field.values[:, mask])**2,
                          vmin=0, vmax=10, cmap="magma", shading="auto", rasterized=True)
    ax.set(title=rf"$\sigma$ = {width}", xlabel=r"Координата $\theta$", ylim=(0, 400))
axes[0].set_ylabel("Время t")
fig.colorbar(image, ax=axes, label=r"Интенсивность $|\psi|^2$", extend="max")
export(fig, "dynamics-a28")

## Слабые волны

Спектральный эталон эксперимента 05 для A = 1.6 и sigma = 0.3293312. Фон строим как пространственно однородное решение того же уравнения из нулевого поля. Относительную частоту определяем как производную разности фаз по времени. Цветовую шкалу ограничиваем для видимости слабых полос.

In [ ]:
ref = load_evolution(runs / "2026-11-20_11-10-40_241883")
window = (ref.times >= 0.49) & (ref.times <= 4.02)
t = ref.times[window]
psi = ref.values[window]

def homogeneous_rhs(t, psi):
    return 2 - (1 + 3j) * psi + 1j * abs(psi)**2 * psi

bg = solve_ivp(homogeneous_rhs, (0, t[-1]), [0j], t_eval=t, rtol=1e-10, atol=1e-12)
phase = np.unwrap(np.angle(psi * bg.y[0].conj()[:, None]), axis=0)
frequency = np.gradient(phase, t, axis=0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True, layout="constrained")
for ax, values, options, title in zip(
    axes, (abs(psi), frequency),
    (dict(cmap="viridis", norm=LogNorm(0.25, 4)), dict(cmap="RdBu_r", vmin=-0.75, vmax=0.75)),
    (r"Амплитуда $|\psi|$", "Частота относительно фона"),
):
    im = ax.pcolormesh(ref.theta, t, values, shading="auto", rasterized=True, **options)
    fig.colorbar(im, ax=ax)
    ax.set(title=title, xlabel=r"Координата $\theta$", ylim=(0.5, 4))
axes[0].set_ylabel("Время t")
export(fig, "waves")

## Проверка PINN

Эксперимент 05. Сравниваем сохраненные поля на одной сетке. Относительная L2-ошибка вычисляется по комплексному полю для каждого момента времени.

In [ ]:
pinn = load_evolution(runs / "2026-11-20_11-18-26_476733")
np.testing.assert_array_equal(pinn.theta, ref.theta)
np.testing.assert_allclose(pinn.times, ref.times, rtol=0, atol=1e-10)
error = 100 * relative_l2_error(pinn.values, ref.values)
fig = plt.figure(figsize=(10.4, 4.25), layout="constrained")
grid = fig.add_gridspec(1, 2, width_ratios=[1.12, 1], wspace=.11)
left = fig.add_subfigure(grid[0])
axes = left.subplots(2, 2, sharex=True, sharey=True)
left.supxlabel(r"Координата $\theta$", fontsize=11)
left.supylabel(r"Интенсивность $|\psi|^2$", fontsize=11)
curve = fig.add_subplot(grid[1])
curve.plot(ref.times, error, color=teal, lw=2)
curve.set(xlim=(-.08, 5.17), ylim=(0, 1.39), xlabel="Время t",
          ylabel="Относительная ошибка, %", title="Ошибка комплексного поля")
for ax, time in zip(axes.flat, (1.48, 2.50, 3.76, 5.00)):
    j = np.abs(ref.times - time).argmin()
    label = f"{time:.2f}".replace(".", ",")
    ax.plot(ref.theta, ref[j].intensity.values, color=gray, lw=2.8, label="SSFM")
    ax.plot(pinn.theta, pinn[j].intensity.values, "--", color=orange, lw=1.6, label="PINN")
    ax.set(xlim=(-np.pi, np.pi), ylim=(0, 13), xticks=[-2, 0, 2],
           yticks=[0, 5, 10], title=f"t = {label}")
    curve.scatter(time, error[j], color=orange, s=24, zorder=3)
    curve.annotate(label, (time, error[j]), xytext=(0, 8), textcoords="offset points",
                   ha="right" if time == 5 else "center", color=orange, fontsize=10)
for ax in [*axes.flat, curve]:
    ax.grid(axis="y", alpha=.17)
axes[0, 0].legend(frameon=False, fontsize=8, loc="upper left", handlelength=1.4)
export(fig, "pinn")

## Адаптивный отбор точек при T = 4

Парное сравнение из эксперимента 06. В обоих запусках 8192 точки, 500000 шагов, обновление каждые 100 шагов и seed 42. Кривая показывает максимум интенсивности по координате на каждом снимке.

In [ ]:
from datetime import datetime
import json

reference = load_evolution(runs / "2026-11-10_07-44-12_914080")
fig, ax = plt.subplots(figsize=(5.6, 3.35), layout="constrained")
ax.plot(reference.times, reference.intensity.values.max(axis=1), color=gray, lw=3, label="SSFM")
for run_id, label, color, style in (
    ("2026-11-21_17-46-40_451027", "Равномерно", orange, "-."),
    ("2026-11-21_18-57-38_683947", "RAD", teal, "--"),
):
    field = load_evolution(runs / run_id)
    np.testing.assert_array_equal(field.theta, reference.theta)
    np.testing.assert_allclose(field.times, reference.times, rtol=0, atol=1e-10)
    ax.plot(field.times, field.intensity.values.max(axis=1), style, color=color, lw=2, label=label)
    error = relative_l2_error(field.values, reference.values).max()
    meta = json.loads((runs / run_id / "metadata.json").read_text(encoding="utf8"))
    elapsed = datetime.fromisoformat(meta["finished_utc"]) - datetime.fromisoformat(meta["started_utc"])
    print(f"{label}: {error:.2%}, {elapsed.total_seconds()/60:.1f} мин")
ax.set(xlabel="Время t", ylabel=r"Пиковая интенсивность $\max_\theta |\psi|^2$", xlim=(0, 4), ylim=(0, 13))
ax.legend(frameon=False, loc="upper right", fontsize=10)
export(fig, "rad-peaks")